# Day 3 - Conversational AI - aka Chatbot!

In [1]:
# imports

import os
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr

In [2]:
# Load environment variables in a file called .env
# Print the key prefixes to help with any debugging

load_dotenv(override=True)
openai_api_key = os.getenv('OPENAI_API_KEY')
anthropic_api_key = os.getenv('ANTHROPIC_API_KEY')
google_api_key = os.getenv('GOOGLE_API_KEY')
deepseek_api_key = os.getenv('DEEPSEEK_API_KEY')

if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")
    
if anthropic_api_key:
    print(f"Anthropic API Key exists and begins {anthropic_api_key[:7]}")
else:
    print("Anthropic API Key not set")

if google_api_key:
    print(f"Google API Key exists and begins {google_api_key[:8]}")
else:
    print("Google API Key not set")

if deepseek_api_key:
    print(f"Deepseek API Key exists and begins {deepseek_api_key[:8]}")
else:
    print("Deepseek API Key not set")

OpenAI API Key exists and begins sk-proj-
Anthropic API Key exists and begins sk-ant-
Google API Key exists and begins AIzaSyBr
Deepseek API Key exists and begins sk-037f2


In [3]:
# Initialize

openai = OpenAI()
MODEL = 'gpt-4o-mini'

In [4]:
def test_user_prompts():
    return [
        "Competition P2019. Group 1. Pitch 4. BK-46 vs EIF has just kicked off.",
        "BK-46 scores.",
        #"BK-46 scores.",
        "BK-46 scores.",
        "EIF scores.",
        #"EIF scores.",
        #"EIF scores.",
        "Game Over."
    ]

system_message = "You are a football commentator."
system_message += " Your audience is parents and kids"
system_message += " You provide funny, light hearted, playful, encouraging, impartial updates to games based on requests."
system_message += " Keep your responses to max 150 characters including emojis."
system_message += " Your response is a single line of text and always in markdown."
system_message += " Always name the competition, pitch and teams."
system_message += " Include the game score if not kickoff."
system_message += " Always include the game score if game over."


# Please read this! A change from the video:

In the video, I explain how we now need to write a function called:

`chat(message, history)`

Which expects to receive `history` in a particular format, which we need to map to the OpenAI format before we call OpenAI:

```
[
    {"role": "system", "content": "system message here"},
    {"role": "user", "content": "first user prompt here"},
    {"role": "assistant", "content": "the assistant's response"},
    {"role": "user", "content": "the new user prompt"},
]
```

But Gradio has been upgraded! Now it will pass in `history` in the exact OpenAI format, perfect for us to send straight to OpenAI.

So our work just got easier!

We will write a function `chat(message, history)` where:  
**message** is the prompt to use  
**history** is the past conversation, in OpenAI format  

We will combine the system message, history and latest message, then call OpenAI.

In [5]:
# Simpler than in my video - we can easily create this function that calls OpenAI
# It's now just 1 line of code to prepare the input to OpenAI!

def chat(message, history):
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]

    print("History is:")
    print(history)
    print("And messages is:")
    print(messages)

    stream = openai.chat.completions.create(model=MODEL, messages=messages, stream=True)

    response = ""
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        yield response

    print("Response is:")
    print(response)

## And then enter Gradio's magic!

In [6]:
gr.ChatInterface(fn=chat, type="messages").launch(share=True)

* Running on local URL:  http://127.0.0.1:7861

Could not create share link. Please check your internet connection or our status page: https://status.gradio.app.


History is:
[]
And messages is:
[{'role': 'system', 'content': 'You are a football commentator. Your audience is parents and kids You provide funny, light hearted, playful, encouraging, impartial updates to games based on requests. Keep your responses to max 150 characters including emojis. Your response is a single line of text and always in markdown. Always name the competition, pitch and teams. Include the game score if not kickoff. Always include the game score if game over.'}, {'role': 'user', 'content': 'ompetition P2019. Group 1. Pitch 4. BK-46 vs EIF has just kicked off.'}]
Response is:
**P2019, Pitch 4:** ⚽️ Kickoff! It's BK-46 vs EIF. Let the fun and excitement begin! 🎉🏟️
History is:
[{'role': 'user', 'metadata': None, 'content': 'ompetition P2019. Group 1. Pitch 4. BK-46 vs EIF has just kicked off.', 'options': None}, {'role': 'assistant', 'metadata': None, 'content': "**P2019, Pitch 4:** ⚽️ Kickoff! It's BK-46 vs EIF. Let the fun and excitement begin! 🎉🏟️", 'options': None}